In [ ]:
from pathlib import Path
import json
import hashlib
import math
import re
import warnings
from itertools import combinations

import numpy as np
import pandas as pd

from scipy.stats import (
    friedmanchisquare,
    wilcoxon,
    spearmanr,
    kendalltau,
)
from scipy.stats import qmc

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    log_loss,
    brier_score_loss,
    roc_auc_score,
)

from xgboost import XGBRegressor
import pulp
import matplotlib.pyplot as plt

GLOBAL_SEED = 42
BOOTSTRAP_SEED = 123

DEVELOPMENT_SEASONS = [2021, 2022, 2023]
DECISION_CALIBRATION_SEASON = 2024
OUT_OF_TIME_SEASON = 2025
EXPERIMENT_SEASONS = DEVELOPMENT_SEASONS + [
    DECISION_CALIBRATION_SEASON,
    OUT_OF_TIME_SEASON,
]

N_RF_CANDIDATES = 64
N_XGB_CANDIDATES = 64
WEIGHT_GRID_STEP = 0.05
N_BOOTSTRAP = 5000
NEAR_OPTIMAL_TOLERANCE = 0.01

BUDGET_MAX = 110.0

def find_repo_root(start=None):
    """Locate the FAME repository root from the current working directory."""
    start = Path.cwd() if start is None else Path(start).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").exists()
            and (candidate / "results").exists()
        ):
            return candidate
    raise RuntimeError(
        "FAME repository root not found. Run this notebook from inside a clone "
        "of the FAME repository."
    )

REPO_ROOT = find_repo_root()
RAW_DATA_DIR = REPO_ROOT / "data" / "raw"
REPRODUCED_DIR = REPO_ROOT / "reproduced"
REPRODUCED_DIR.mkdir(parents=True, exist_ok=True)

ROOT = REPRODUCED_DIR / "fantasy_football" / "primary"
MODEL_DIR = ROOT / "01_modelagem_tuning"
PRED_DIR = ROOT / "02_previsoes_walkforward"
DECISION_DIR = ROOT / "03_calibracao_decisao"
RESULTS_DIR = ROOT / "04_resultados_2025"
FIG_DIR = ROOT / "05_figuras"
AUDIT_DIR = ROOT / "06_auditoria"
CHECKPOINT_DIR = ROOT / "07_checkpoints"

for d in [
    ROOT, MODEL_DIR, PRED_DIR, DECISION_DIR, RESULTS_DIR,
    FIG_DIR, AUDIT_DIR, CHECKPOINT_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

POSICOES = {
    1: "goleiro",
    2: "lateral",
    3: "zagueiro",
    4: "meia",
    5: "atacante",
    6: "tecnico",
}
MAPA_POSICOES = {v: k for k, v in POSICOES.items()}

FORMACOES = {
    "3-4-3": {"goleiro": 1, "lateral": 0, "zagueiro": 3, "meia": 4, "atacante": 3, "tecnico": 1},
    "4-3-3": {"goleiro": 1, "lateral": 2, "zagueiro": 2, "meia": 3, "atacante": 3, "tecnico": 1},
    "4-4-2": {"goleiro": 1, "lateral": 2, "zagueiro": 2, "meia": 4, "atacante": 2, "tecnico": 1},
    "3-5-2": {"goleiro": 1, "lateral": 0, "zagueiro": 3, "meia": 5, "atacante": 2, "tecnico": 1},
    "4-5-1": {"goleiro": 1, "lateral": 2, "zagueiro": 2, "meia": 5, "atacante": 1, "tecnico": 1},
    "5-3-2": {"goleiro": 1, "lateral": 2, "zagueiro": 3, "meia": 3, "atacante": 2, "tecnico": 1},
    "5-4-1": {"goleiro": 1, "lateral": 2, "zagueiro": 3, "meia": 4, "atacante": 1, "tecnico": 1},
}

FAME_ORIGINAL_WEIGHTS = (0.70, 0.20, 0.10)
EQUAL_WEIGHTS = (1/3, 1/3, 1/3)

print("Diretório de saída:", ROOT.resolve())


In [ ]:
import os

DATA_FILE_OVERRIDE = os.environ.get("FAME_CARTOLA_DATA")
DEFAULT_DATA_FILE = (
    RAW_DATA_DIR
    / "fantasy_football"
    / "cartola_base_modelagem_2021_2026.csv"
)

def localizar_base():
    candidate = Path(DATA_FILE_OVERRIDE).expanduser() if DATA_FILE_OVERRIDE else DEFAULT_DATA_FILE
    if not candidate.exists():
        raise FileNotFoundError(
            "Fantasy-football analytical dataset not found.\n"
            f"Expected: {DEFAULT_DATA_FILE}\n"
            "Alternatively set environment variable FAME_CARTOLA_DATA to the file path.\n"
            "See data/SOURCES_AND_RECONSTRUCTION.md."
        )
    return candidate.resolve()

DATA_FILE = localizar_base()
cartola = pd.read_csv(DATA_FILE)

cartola = cartola[cartola["temporada"].isin(EXPERIMENT_SEASONS)].copy()

required = [
    "temporada", "rodada_id", "atleta_id", "posicao_id",
    "pontos_num", "preco_num", "media_num", "jogos_num",
]
missing_required = [c for c in required if c not in cartola.columns]
if missing_required:
    raise KeyError(f"Variáveis obrigatórias ausentes: {missing_required}")

for c in [
    "temporada","rodada_id","atleta_id","posicao_id","status_id",
    "pontos_num","preco_num","media_num","jogos_num","variacao_num","clube_id"
]:
    if c in cartola.columns:
        cartola[c] = pd.to_numeric(cartola[c], errors="coerce")

key = ["atleta_id","temporada","rodada_id"]
dups = cartola.duplicated(key, keep=False)
if dups.any():
    display(cartola.loc[dups, key].head(20))
    raise AssertionError(
        f"Foram encontradas {int(dups.sum())} linhas duplicadas por atleta-temporada-rodada."
    )

cartola = cartola.sort_values(key).reset_index(drop=True)

seasons = sorted(cartola["temporada"].dropna().astype(int).unique())
if set(EXPERIMENT_SEASONS) != set(seasons):
    raise AssertionError(
        f"Esperadas temporadas {EXPERIMENT_SEASONS}; encontradas {seasons}."
    )

audit_input = pd.DataFrame([{

    "data_file": "data/raw/fantasy_football/cartola_base_modelagem_2021_2026.csv",
    "n_rows": len(cartola),
    "n_columns": cartola.shape[1],
    "seasons": ",".join(map(str,seasons)),
    "n_players": int(cartola["atleta_id"].nunique()),
    "budget_max": BUDGET_MAX,
    "n_formations": len(FORMACOES),
    "rows_2026_used": 0,
}])
audit_input.to_csv(AUDIT_DIR/"input_data_audit.csv", index=False, encoding="utf-8-sig")
display(audit_input)


In [ ]:
CONSISTENT_SCOUT_CODES = {
    "A","CA","CV","DD","DP","FC","FD","FF","FS","FT",
    "G","GC","GS","I","PE","PP","RB","SG","DS","DE","PS","PC","VC"
}

def localizar_fonte_scout(df, code):
    candidates = [code, f"scout.{code}", f"scout_{code}"]
    existentes = [c for c in candidates if c in df.columns]

    if not existentes:
        return None

    coverage = {
        c: pd.to_numeric(df[c], errors="coerce").notna().mean()
        for c in existentes
    }
    return max(coverage, key=coverage.get)

SCOUT_SOURCE_MAP = {}

for code in sorted(CONSISTENT_SCOUT_CODES):
    src = localizar_fonte_scout(cartola, code)
    if src is not None:
        SCOUT_SOURCE_MAP[code] = src

print("Scouts canônicos identificados:")
print(SCOUT_SOURCE_MAP)

scout_round_cols = []

for code, src in SCOUT_SOURCE_MAP.items():
    cum_col = f"{code}_cum"
    round_col = f"{code}_round"

    cartola[cum_col] = (
        pd.to_numeric(cartola[src], errors="coerce")
        .fillna(0.0)
    )

    diff = (
        cartola
        .groupby(["atleta_id","temporada"], sort=False)[cum_col]
        .diff()
    )

    first_mask = (
        cartola
        .groupby(["atleta_id","temporada"])
        .cumcount()
        .eq(0)
    )

    cartola[round_col] = diff
    cartola.loc[first_mask, round_col] = cartola.loc[first_mask, cum_col]

    cartola[round_col] = cartola[round_col].clip(lower=0)

    scout_round_cols.append(round_col)

cartola = (
    cartola
    .sort_values(["temporada","atleta_id","rodada_id"])
    .reset_index(drop=True)
)

g = cartola.groupby(["atleta_id","temporada"], sort=False)

cartola["_jogos_lag1"] = g["jogos_num"].shift(1)
cartola["_played_current"] = np.nan

mask_transition_observable = (
    cartola["_jogos_lag1"].notna()
    & cartola["jogos_num"].notna()
)

cartola.loc[mask_transition_observable, "_played_current"] = (
    cartola.loc[mask_transition_observable, "jogos_num"]
    >
    cartola.loc[mask_transition_observable, "_jogos_lag1"]
).astype(float)

base_numeric = [
    c for c in [
        "pontos_num",
        "preco_num",
        "variacao_num",
        "media_num",
        "jogos_num",
    ]
    if c in cartola.columns
] + scout_round_cols

feature_parts = {}

for var in base_numeric:
    feature_parts[f"{var}_lag1"] = g[var].shift(1)

    feature_parts[f"{var}_hist_mean"] = (
        cartola
        .groupby(["atleta_id","temporada"])[var]
        .transform(
            lambda s: s.shift(1).expanding(min_periods=1).mean()
        )
    )

cartola["_points_if_played"] = (
    cartola["pontos_num"]
    .where(cartola["_played_current"].eq(1))
)

feature_parts["played_points_hist_mean"] = (
    cartola
    .groupby(["atleta_id","temporada"])["_points_if_played"]
    .transform(
        lambda s: s.shift(1).expanding(min_periods=1).mean()
    )
)

feature_parts["played_points_hist_std"] = (
    cartola
    .groupby(["atleta_id","temporada"])["_points_if_played"]
    .transform(
        lambda s: s.shift(1).expanding(min_periods=2).std()
    )
)

feature_parts["participation_hist_rate"] = (
    cartola
    .groupby(["atleta_id","temporada"])["_played_current"]
    .transform(
        lambda s: s.shift(1).expanding(min_periods=1).mean()
    )
)

features_hist = pd.DataFrame(
    feature_parts,
    index=cartola.index
)

cartola = pd.concat(
    [cartola, features_hist],
    axis=1
)

cartola["historical_unconditional_baseline"] = (
    cartola
    .groupby(["atleta_id","temporada"])["pontos_num"]
    .transform(
        lambda s: s.shift(1).expanding(min_periods=1).mean()
    )
)

cartola["historical_conditional_baseline"] = (
    cartola["played_points_hist_mean"]
)

scout_audit_rows = []

for code, src in SCOUT_SOURCE_MAP.items():
    for season in EXPERIMENT_SEASONS:
        d = cartola[cartola["temporada"].eq(season)]

        scout_audit_rows.append({
            "scout": code,
            "source_column": src,
            "season": season,
            "n_rows": len(d),
            "cum_nonmissing_rate": float(d[f"{code}_cum"].notna().mean()),
            "round_mean": float(d[f"{code}_round"].mean()),
            "round_positive_rate": float((d[f"{code}_round"] > 0).mean()),
        })

scout_audit = pd.DataFrame(scout_audit_rows)

scout_audit.to_csv(
    AUDIT_DIR / "scout_consistency_audit_2021_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame({
    "scout": list(SCOUT_SOURCE_MAP.keys()),
    "source_column": list(SCOUT_SOURCE_MAP.values()),
}).to_csv(
    AUDIT_DIR / "scout_source_map.csv",
    index=False,
    encoding="utf-8-sig"
)

feature_audit = pd.DataFrame([{
    "n_consistent_scouts": len(SCOUT_SOURCE_MAP),
    "n_scout_round_features": len(scout_round_cols),
    "n_historical_features": len(features_hist.columns),
    "first_observation_participation_is_nan": bool(
        cartola
        .groupby(["atleta_id","temporada"])
        .head(1)["_played_current"]
        .isna()
        .all()
    ),
}])

feature_audit.to_csv(
    AUDIT_DIR / "feature_construction_audit.csv",
    index=False,
    encoding="utf-8-sig"
)

display(feature_audit)


In [ ]:
future = cartola[
    ["atleta_id","temporada","rodada_id","pontos_num","jogos_num"]
].copy().rename(columns={
    "rodada_id":"rodada_target",
    "pontos_num":"target_next_observed",
    "jogos_num":"jogos_next",
})

cartola["rodada_target"] = cartola["rodada_id"] + 1
cartola = cartola.merge(
    future,
    on=["atleta_id","temporada","rodada_target"],
    how="left",
    validate="one_to_one",
)

valid_decision_row = cartola["rodada_id"].between(1,37)

cartola["has_next_market_row"] = cartola["target_next_observed"].notna()
cartola["target_next"] = cartola["target_next_observed"]

cartola.loc[
    valid_decision_row & cartola["target_next"].isna(),
    "target_next"
] = 0.0

cartola["participated_next"] = np.nan
mask_current_games_known = valid_decision_row & cartola["jogos_num"].notna()
cartola.loc[mask_current_games_known, "participated_next"] = 0.0

mask_next_games_known = mask_current_games_known & cartola["jogos_next"].notna()
cartola.loc[mask_next_games_known, "participated_next"] = (
    cartola.loc[mask_next_games_known, "jogos_next"]
    > cartola.loc[mask_next_games_known, "jogos_num"]
).astype(float)

cartola.loc[cartola["posicao_id"].eq(6), "participated_next"] = 1.0

target_audit = (
    cartola.loc[valid_decision_row]
    .groupby("temporada",as_index=False)
    .agg(
        n_decision_rows=("atleta_id","size"),
        pct_next_market_row=("has_next_market_row","mean"),
        pct_missing_next_market=("has_next_market_row",lambda s: 1-float(s.mean())),
        pct_zero_target=("target_next",lambda s: float((s==0).mean())),
        pct_participated_next=("participated_next","mean"),
    )
)
target_audit.to_csv(AUDIT_DIR/"target_alignment_audit.csv",index=False,encoding="utf-8-sig")
display(target_audit)


In [ ]:
candidate_numeric = [
    c for c in cartola.columns
    if (
        c.endswith("_lag1")
        or c.endswith("_hist_mean")
        or c in [
            "played_points_hist_std",
            "participation_hist_rate",
            "preco_num",
            "variacao_num",
            "media_num",
            "jogos_num",
        ]
    )
]

forbidden = {
    "target_next",
    "target_next_observed",
    "jogos_next",
    "participated_next",
    "rodada_target",
    "pontos_num",
    "_played_current",
    "_jogos_lag1",
    "_points_if_played",
}

candidate_numeric = [
    c for c in candidate_numeric
    if not c.endswith("_cum")
]

features_numericas = sorted(
    [
        c for c in set(candidate_numeric)
        if c not in forbidden
    ]
)

features_categoricas = [
    c for c in ["clube_id","status_id"]
    if c in cartola.columns
]

availability_categoricas = [
    c for c in ["clube_id","status_id","posicao_id"]
    if c in cartola.columns
]

features = features_numericas + features_categoricas
availability_features = features_numericas + availability_categoricas

def make_regression_preprocessor():
    return ColumnTransformer([
        (
            "num",
            SimpleImputer(strategy="median"),
            features_numericas
        ),
        (
            "cat",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "onehot",
                    OneHotEncoder(handle_unknown="ignore")
                ),
            ]),
            features_categoricas
        ),
    ])

def make_availability_preprocessor():
    return ColumnTransformer([
        (
            "num",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="median")
                ),
                (
                    "scale",
                    StandardScaler()
                ),
            ]),
            features_numericas
        ),
        (
            "cat",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "onehot",
                    OneHotEncoder(handle_unknown="ignore")
                ),
            ]),
            availability_categoricas
        ),
    ])

pd.DataFrame({
    "feature": features,
    "role": (
        ["numeric"] * len(features_numericas)
        +
        ["categorical"] * len(features_categoricas)
    )
}).to_csv(
    MODEL_DIR / "predictive_feature_dictionary.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame({
    "feature": availability_features,
    "role": (
        ["numeric"] * len(features_numericas)
        +
        ["categorical"] * len(availability_categoricas)
    )
}).to_csv(
    MODEL_DIR / "availability_feature_dictionary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("N features preditivas:", len(features))
print("N features disponibilidade:", len(availability_features))


In [ ]:
TEMPORAL_FOLDS = [
    {
        "fold": 1,
        "train_end": (2021,19),
        "valid_start": (2021,20),
        "valid_end": (2021,38),
    },
    {
        "fold": 2,
        "train_end": (2021,38),
        "valid_start": (2022,2),
        "valid_end": (2022,19),
    },
    {
        "fold": 3,
        "train_end": (2022,19),
        "valid_start": (2022,20),
        "valid_end": (2022,38),
    },
    {
        "fold": 4,
        "train_end": (2022,38),
        "valid_start": (2023,2),
        "valid_end": (2023,19),
    },
    {
        "fold": 5,
        "train_end": (2023,19),
        "valid_start": (2023,20),
        "valid_end": (2023,38),
    },
]

def temporal_key(season, target_round):
    return int(season)*100 + int(target_round)

cartola["_target_time_key"] = (
    cartola["temporada"].astype(int)*100
    + cartola["rodada_target"].fillna(99).astype(int)
)

def split_fold(df, fold):
    train_end_key = temporal_key(*fold["train_end"])
    valid_start_key = temporal_key(*fold["valid_start"])
    valid_end_key = temporal_key(*fold["valid_end"])

    train = df[
        df["_target_time_key"].le(train_end_key)
        & df["rodada_id"].between(1,37)
    ].copy()

    valid = df[
        df["_target_time_key"].between(valid_start_key,valid_end_key)
        & df["rodada_id"].between(1,37)
    ].copy()

    return train,valid

fold_audit = []
for f in TEMPORAL_FOLDS:
    tr,va = split_fold(cartola,f)
    fold_audit.append({
        "fold":f["fold"],
        "train_end":str(f["train_end"]),
        "valid_start":str(f["valid_start"]),
        "valid_end":str(f["valid_end"]),
        "n_train":len(tr),
        "n_valid":len(va),
        "max_train_key":tr["_target_time_key"].max(),
        "min_valid_key":va["_target_time_key"].min(),
        "strict_temporal_order":bool(
            tr["_target_time_key"].max() < va["_target_time_key"].min()
        ),
    })

fold_audit = pd.DataFrame(fold_audit)
fold_audit.to_csv(MODEL_DIR/"temporal_development_folds.csv",index=False,encoding="utf-8-sig")
display(fold_audit)


In [ ]:
AVAILABILITY_C_GRID = np.logspace(-4,3,15)

def calibration_diagnostics(y,p,n_bins=10):
    y = np.asarray(y,dtype=int)
    p = np.clip(np.asarray(p,dtype=float),1e-6,1-1e-6)

    z = np.log(p/(1-p)).reshape(-1,1)
    recal = LogisticRegression(
        C=1e6,
        penalty="l2",
        solver="liblinear",
        max_iter=2000,
        random_state=GLOBAL_SEED,
    )
    recal.fit(z,y)
    intercept = float(recal.intercept_[0])
    slope = float(recal.coef_[0,0])

    bins = np.linspace(0,1,n_bins+1)
    bin_id = np.clip(np.digitize(p,bins,right=False)-1,0,n_bins-1)

    reliability = []
    ece = 0.0

    for b in range(n_bins):
        mask = bin_id == b
        if not mask.any():
            continue
        mean_p = float(p[mask].mean())
        obs = float(y[mask].mean())
        weight = float(mask.mean())
        ece += weight*abs(mean_p-obs)
        reliability.append({
            "bin":b,
            "n":int(mask.sum()),
            "mean_predicted_probability":mean_p,
            "observed_frequency":obs,
        })

    return intercept,slope,float(ece),pd.DataFrame(reliability)

availability_rows=[]
availability_reliability_parts=[]

for C in AVAILABILITY_C_GRID:
    for f in TEMPORAL_FOLDS:
        train,valid = split_fold(cartola,f)

        train = train[
            train["participated_next"].notna()
            & train["posicao_id"].ne(6)
        ].copy()

        valid = valid[
            valid["participated_next"].notna()
            & valid["posicao_id"].ne(6)
        ].copy()

        if len(train)<100 or len(valid)<50:
            continue

        model = Pipeline([
            ("prep",make_availability_preprocessor()),
            ("model",LogisticRegression(
                C=float(C),
                penalty="l2",
                solver="liblinear",
                max_iter=2000,
                random_state=GLOBAL_SEED,
            )),
        ])

        model.fit(train[availability_features],train["participated_next"].astype(int))
        p = model.predict_proba(valid[availability_features])[:,1]
        y = valid["participated_next"].astype(int).to_numpy()

        intercept,slope,ece,rel = calibration_diagnostics(y,p)

        availability_rows.append({
            "C":float(C),
            "fold":f["fold"],
            "n":len(valid),
            "log_loss":log_loss(y,p,labels=[0,1]),
            "brier":brier_score_loss(y,p),
            "roc_auc":roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan,
            "calibration_intercept":intercept,
            "calibration_slope":slope,
            "ece_10_bins":ece,
        })

        rel["C"]=float(C)
        rel["fold"]=f["fold"]
        availability_reliability_parts.append(rel)

availability_cv = pd.DataFrame(availability_rows)

availability_summary = (
    availability_cv.groupby("C",as_index=False)
    .agg(
        mean_log_loss=("log_loss","mean"),
        mean_brier=("brier","mean"),
        mean_roc_auc=("roc_auc","mean"),
        mean_calibration_intercept=("calibration_intercept","mean"),
        mean_calibration_slope=("calibration_slope","mean"),
        mean_ece=("ece_10_bins","mean"),
    )
    .sort_values(["mean_log_loss","mean_brier","C"])
    .reset_index(drop=True)
)

BEST_AVAILABILITY_C = float(availability_summary.iloc[0]["C"])

availability_reliability = pd.concat(
    availability_reliability_parts,ignore_index=True
)

availability_cv.to_csv(MODEL_DIR/"availability_temporal_cv.csv",index=False,encoding="utf-8-sig")
availability_summary.to_csv(MODEL_DIR/"availability_tuning_summary.csv",index=False,encoding="utf-8-sig")
availability_reliability.to_csv(MODEL_DIR/"availability_reliability_all_candidates.csv",index=False,encoding="utf-8-sig")

print("C selecionado:",BEST_AVAILABILITY_C)
display(availability_summary.head(10))


In [ ]:
def sobol_candidates_rf(n=N_RF_CANDIDATES,seed=GLOBAL_SEED):
    sampler=qmc.Sobol(d=4,scramble=True,seed=seed)
    m=int(np.ceil(np.log2(n)))
    u=sampler.random_base2(m=m)[:n]

    depth_choices=[2,3,5,8,12,None]
    leaf_choices=[2,5,10,20,30,50,75]
    maxfeat_choices=["sqrt",0.3,0.5,0.8,1.0]

    out=[]
    for r in u:
        n_estimators=int(max(200,round((300+r[0]*1500)/50)*50))
        out.append({
            "n_estimators":n_estimators,
            "max_depth":depth_choices[min(int(r[1]*len(depth_choices)),len(depth_choices)-1)],
            "min_samples_leaf":leaf_choices[min(int(r[2]*len(leaf_choices)),len(leaf_choices)-1)],
            "max_features":maxfeat_choices[min(int(r[3]*len(maxfeat_choices)),len(maxfeat_choices)-1)],
        })
    return out

def sobol_candidates_xgb(n=N_XGB_CANDIDATES,seed=GLOBAL_SEED+1):
    sampler=qmc.Sobol(d=7,scramble=True,seed=seed)
    m=int(np.ceil(np.log2(n)))
    u=sampler.random_base2(m=m)[:n]

    out=[]
    for r in u:
        out.append({
            "n_estimators":int(max(100,round((200+r[0]*1400)/50)*50)),
            "max_depth":int(2+np.floor(r[1]*7)),
            "learning_rate":float(10**(-3+r[2]*2)),
            "subsample":float(0.5+0.5*r[3]),
            "colsample_bytree":float(0.5+0.5*r[4]),
            "reg_alpha":float(10**(-4+r[5]*5.3)),
            "reg_lambda":float(10**(-3+r[6]*4.7)),
        })
    return out

RF_CANDIDATES=sobol_candidates_rf()
XGB_CANDIDATES=sobol_candidates_xgb()

pd.DataFrame(RF_CANDIDATES).to_csv(MODEL_DIR/"rf_search_candidates.csv",index=False,encoding="utf-8-sig")
pd.DataFrame(XGB_CANDIDATES).to_csv(MODEL_DIR/"xgb_search_candidates.csv",index=False,encoding="utf-8-sig")

print("RF candidates:",len(RF_CANDIDATES))
print("XGB candidates:",len(XGB_CANDIDATES))


In [ ]:
def regression_metrics(y,pred):
    y=np.asarray(y,dtype=float)
    pred=np.asarray(pred,dtype=float)
    ok=np.isfinite(y)&np.isfinite(pred)
    y,pred=y[ok],pred[ok]

    if len(y)==0:
        return {"n":0,"MAE":np.nan,"RMSE":np.nan,"R2":np.nan,"correlation":np.nan}

    return {
        "n":int(len(y)),
        "MAE":float(mean_absolute_error(y,pred)),
        "RMSE":float(np.sqrt(mean_squared_error(y,pred))),
        "R2":float(r2_score(y,pred)) if len(y)>1 else np.nan,
        "correlation":float(np.corrcoef(y,pred)[0,1]) if len(y)>1 else np.nan,
    }

def build_rf(params):
    return Pipeline([
        ("prep",make_regression_preprocessor()),
        ("model",RandomForestRegressor(
            **params,
            random_state=GLOBAL_SEED,
            n_jobs=-1,
        )),
    ])

def build_xgb(params):
    return Pipeline([
        ("prep",make_regression_preprocessor()),
        ("model",XGBRegressor(
            **params,
            objective="reg:squarederror",
            random_state=GLOBAL_SEED,
            n_jobs=-1,
        )),
    ])

def tuning_checkpoint_path(model_name,posicao_id):
    return CHECKPOINT_DIR/f"tuning_{model_name.lower()}_pos_{posicao_id}.csv"

def tune_model_for_position(posicao_id,model_name,candidates):
    checkpoint=tuning_checkpoint_path(model_name,posicao_id)

    if checkpoint.exists():
        detail=pd.read_csv(checkpoint)
        print(f"Checkpoint carregado: {checkpoint.name}")
    else:
        rows=[]

        for cid,params in enumerate(candidates,start=1):
            for f in TEMPORAL_FOLDS:
                train,valid=split_fold(cartola,f)

                train=train[
                    train["participated_next"].eq(1)
                    & train["posicao_id"].eq(posicao_id)
                    & train["target_next"].notna()
                ].copy()

                valid=valid[
                    valid["participated_next"].eq(1)
                    & valid["posicao_id"].eq(posicao_id)
                    & valid["target_next"].notna()
                ].copy()

                if len(train)<50 or len(valid)<20:
                    continue

                model=build_rf(params) if model_name=="RF" else build_xgb(params)
                model.fit(train[features],train["target_next"])
                pred=model.predict(valid[features])
                m=regression_metrics(valid["target_next"],pred)

                rows.append({
                    "position_id":posicao_id,
                    "position":POSICOES[posicao_id],
                    "model":model_name,
                    "candidate_id":cid,
                    "fold":f["fold"],
                    **m,
                    "params_json":json.dumps(params,sort_keys=True),
                })

        detail=pd.DataFrame(rows)

        if detail.empty:
            raise RuntimeError(
                f"Sem resultados para {model_name}, posição {posicao_id}."
            )

        detail.to_csv(checkpoint,index=False,encoding="utf-8-sig")

    summary=(
        detail.groupby(
            ["position_id","position","model","candidate_id","params_json"],
            as_index=False
        )
        .agg(
            folds=("fold","nunique"),
            mean_RMSE=("RMSE","mean"),
            mean_MAE=("MAE","mean"),
            mean_R2=("R2","mean"),
            mean_correlation=("correlation","mean"),
        )
        .sort_values(["mean_RMSE","mean_MAE","candidate_id"])
        .reset_index(drop=True)
    )

    best=summary.iloc[0]
    return detail,summary,json.loads(best["params_json"])

tuning_detail_parts=[]
tuning_summary_parts=[]
BEST_RF_PARAMS={}
BEST_XGB_PARAMS={}

for posicao_id in sorted(POSICOES):
    print("\nPosição:",POSICOES[posicao_id])

    d,s,best=tune_model_for_position(posicao_id,"RF",RF_CANDIDATES)
    tuning_detail_parts.append(d)
    tuning_summary_parts.append(s)
    BEST_RF_PARAMS[posicao_id]=best

    d,s,best=tune_model_for_position(posicao_id,"XGB",XGB_CANDIDATES)
    tuning_detail_parts.append(d)
    tuning_summary_parts.append(s)
    BEST_XGB_PARAMS[posicao_id]=best

tuning_detail=pd.concat(tuning_detail_parts,ignore_index=True)
tuning_summary=pd.concat(tuning_summary_parts,ignore_index=True)

tuning_detail.to_csv(MODEL_DIR/"predictive_tuning_temporal_detail.csv",index=False,encoding="utf-8-sig")
tuning_summary.to_csv(MODEL_DIR/"predictive_tuning_temporal_summary.csv",index=False,encoding="utf-8-sig")

best_rows=[]
for p in sorted(POSICOES):
    best_rows.append({
        "position_id":p,
        "position":POSICOES[p],
        "model":"RF",
        "params_json":json.dumps(BEST_RF_PARAMS[p],sort_keys=True),
    })
    best_rows.append({
        "position_id":p,
        "position":POSICOES[p],
        "model":"XGB",
        "params_json":json.dumps(BEST_XGB_PARAMS[p],sort_keys=True),
    })

frozen_hyperparameters=pd.DataFrame(best_rows)
frozen_hyperparameters.to_csv(MODEL_DIR/"frozen_predictive_hyperparameters.csv",index=False,encoding="utf-8-sig")
display(frozen_hyperparameters)


In [ ]:
oof_rows=[]

for p in sorted(POSICOES):
    for f in TEMPORAL_FOLDS:
        train,valid=split_fold(cartola,f)

        train=train[
            train["participated_next"].eq(1)
            & train["posicao_id"].eq(p)
            & train["target_next"].notna()
        ].copy()

        valid=valid[
            valid["participated_next"].eq(1)
            & valid["posicao_id"].eq(p)
            & valid["target_next"].notna()
        ].copy()

        if len(train)<50 or len(valid)<20:
            continue

        rf=build_rf(BEST_RF_PARAMS[p])
        xgb=build_xgb(BEST_XGB_PARAMS[p])

        rf.fit(train[features],train["target_next"])
        xgb.fit(train[features],train["target_next"])

        prf=rf.predict(valid[features])
        pxgb=xgb.predict(valid[features])

        tmp=pd.DataFrame({
            "position_id":p,
            "position":POSICOES[p],
            "fold":f["fold"],
            "y":valid["target_next"].to_numpy(float),
            "pred_rf":prf,
            "pred_xgb":pxgb,
        })
        oof_rows.append(tmp)

oof_predictions=pd.concat(oof_rows,ignore_index=True)

ENSEMBLE_ALPHA={}

alpha_rows=[]
for p,gp in oof_predictions.groupby("position_id"):
    a=gp["pred_rf"].to_numpy(float)-gp["pred_xgb"].to_numpy(float)
    b=gp["y"].to_numpy(float)-gp["pred_xgb"].to_numpy(float)
    denom=float(np.sum(a*a))

    if denom<=1e-12:
        alpha=0.5
        method="indistinguishable_predictions_fallback"
    else:
        alpha=float(np.clip(np.sum(a*b)/denom,0,1))
        method="closed_form_oof_sse_minimizer"

    ENSEMBLE_ALPHA[int(p)]=alpha
    pred=alpha*gp["pred_rf"].to_numpy()+ (1-alpha)*gp["pred_xgb"].to_numpy()
    m=regression_metrics(gp["y"],pred)

    alpha_rows.append({
        "position_id":int(p),
        "position":POSICOES[int(p)],
        "alpha_rf":alpha,
        "alpha_xgb":1-alpha,
        "method":method,
        **m,
    })

ensemble_weights=pd.DataFrame(alpha_rows)

oof_predictions.to_csv(MODEL_DIR/"development_oof_predictions_rf_xgb.csv",index=False,encoding="utf-8-sig")
ensemble_weights.to_csv(MODEL_DIR/"frozen_ensemble_weights.csv",index=False,encoding="utf-8-sig")

display(ensemble_weights)


In [ ]:
def make_availability_model():
    return Pipeline([
        ("prep",make_availability_preprocessor()),
        ("model",LogisticRegression(
            C=BEST_AVAILABILITY_C,
            penalty="l2",
            solver="liblinear",
            max_iter=2000,
            random_state=GLOBAL_SEED,
        )),
    ])

def predict_one_round_position(season,round_target,posicao_id):
    target_rows=cartola[
        cartola["temporada"].eq(season)
        & cartola["rodada_target"].eq(round_target)
        & cartola["posicao_id"].eq(posicao_id)
    ].copy()

    if target_rows.empty:
        return pd.DataFrame(),None

    train_mask=(
        cartola["rodada_id"].between(1,37)
        & cartola["target_next"].notna()
        & (
            (cartola["temporada"]<season)
            | (
                cartola["temporada"].eq(season)
                & cartola["rodada_target"].lt(round_target)
            )
        )
    )

    train_all=cartola[train_mask].copy()

    train_perf=train_all[
        train_all["participated_next"].eq(1)
        & train_all["posicao_id"].eq(posicao_id)
    ].copy()

    if len(train_perf)<50:
        return pd.DataFrame(),None

    rf=build_rf(BEST_RF_PARAMS[posicao_id])
    xgb=build_xgb(BEST_XGB_PARAMS[posicao_id])

    rf.fit(train_perf[features],train_perf["target_next"])
    xgb.fit(train_perf[features],train_perf["target_next"])

    pred_rf=rf.predict(target_rows[features])
    pred_xgb=xgb.predict(target_rows[features])

    alpha=ENSEMBLE_ALPHA[posicao_id]
    pred_conditional=alpha*pred_rf+(1-alpha)*pred_xgb

    if posicao_id==6:
        p_play=np.ones(len(target_rows),dtype=float)
    else:
        train_av=train_all[
            train_all["participated_next"].notna()
            & train_all["posicao_id"].ne(6)
        ].copy()

        av=make_availability_model()
        av.fit(
            train_av[availability_features],
            train_av["participated_next"].astype(int)
        )
        p_play=av.predict_proba(target_rows[availability_features])[:,1]

    pred_expected=pred_conditional*p_play

    out=target_rows.copy()
    out["pred_rf_conditional"]=pred_rf
    out["pred_xgb_conditional"]=pred_xgb
    out["pred_conditional_ensemble"]=pred_conditional
    out["ensemble_alpha_rf"]=alpha
    out["prob_play_estimated"]=p_play
    out["pred_expected"]=pred_expected

    std_ref=pd.to_numeric(
        train_perf["played_points_hist_std"],errors="coerce"
    ).median()

    if not np.isfinite(std_ref):
        std_ref=0.0

    out["historical_uncertainty"]=(
        pd.to_numeric(out["played_points_hist_std"],errors="coerce")
        .fillna(std_ref)
        .clip(lower=0)
    )

    out["upside_component_raw"]=out["historical_uncertainty"]

    price=pd.to_numeric(out["preco_num"],errors="coerce")
    out["economic_efficiency"]=(
        out["pred_expected"]/price.where(price>0)
    )

    out["pontos_realizados"]=out["target_next"]
    out["prediction_season"]=season
    out["prediction_round"]=round_target

    aud={
        "season":season,
        "round_target":round_target,
        "position_id":posicao_id,
        "position":POSICOES[posicao_id],
        "n_train_performance":len(train_perf),
        "n_predicted":len(out),
        "max_train_time_key":train_all["_target_time_key"].max(),
        "target_time_key":temporal_key(season,round_target),
        "strict_no_future_training":bool(
            train_all["_target_time_key"].max()<temporal_key(season,round_target)
        ),
    }

    return out,aud

prediction_parts=[]
prediction_audit_rows=[]

for season in [2024,2025]:
    rounds=sorted(
        cartola.loc[
            cartola["temporada"].eq(season)
            & cartola["rodada_id"].between(1,37),
            "rodada_target"
        ].dropna().astype(int).unique()
    )

    for r in rounds:
        print(f"{season} round {r}")
        for p in sorted(POSICOES):
            out,aud=predict_one_round_position(season,r,p)
            if not out.empty:
                prediction_parts.append(out)
            if aud is not None:
                prediction_audit_rows.append(aud)

predictions=pd.concat(prediction_parts,ignore_index=True)
prediction_audit=pd.DataFrame(prediction_audit_rows)

predictions.to_csv(PRED_DIR/"walkforward_predictions_2024_2025.csv",index=False,encoding="utf-8-sig")
prediction_audit.to_csv(AUDIT_DIR/"walkforward_leakage_audit.csv",index=False,encoding="utf-8-sig")

if not prediction_audit["strict_no_future_training"].all():
    raise AssertionError("Falha na auditoria temporal walk-forward.")

cal=predictions[predictions["temporada"].eq(2024)].copy()
test=predictions[predictions["temporada"].eq(2025)].copy()

print("2024:",cal.shape,"2025:",test.shape)


In [ ]:
predictive_rows=[]

for season,d in [(2024,cal),(2025,test)]:

    dd=d[d["participated_next"].eq(1)].copy()

    conditional_models={
        "Historical conditional mean baseline":"historical_conditional_baseline",
        "Random Forest":"pred_rf_conditional",
        "XGBoost":"pred_xgb_conditional",
        "OOF-weighted RF/XGB ensemble":"pred_conditional_ensemble",
    }

    for name,col in conditional_models.items():
        m=regression_metrics(dd["pontos_realizados"],dd[col])
        predictive_rows.append({
            "season":season,
            "estimand":"conditional_performance_given_participation",
            "position":"ALL",
            "model":name,
            **m,
        })

        for p,gp in dd.groupby("posicao_id"):
            mm=regression_metrics(gp["pontos_realizados"],gp[col])
            predictive_rows.append({
                "season":season,
                "estimand":"conditional_performance_given_participation",
                "position":POSICOES.get(int(p),str(p)),
                "model":name,
                **mm,
            })

    unconditional_models={
        "Historical unconditional mean baseline":"historical_unconditional_baseline",
        "Availability-adjusted expected prediction":"pred_expected",
    }

    for name,col in unconditional_models.items():
        m=regression_metrics(d["pontos_realizados"],d[col])
        predictive_rows.append({
            "season":season,
            "estimand":"unconditional_realized_points",
            "position":"ALL",
            "model":name,
            **m,
        })

        for p,gp in d.groupby("posicao_id"):
            mm=regression_metrics(gp["pontos_realizados"],gp[col])
            predictive_rows.append({
                "season":season,
                "estimand":"unconditional_realized_points",
                "position":POSICOES.get(int(p),str(p)),
                "model":name,
                **mm,
            })

predictive_metrics=pd.DataFrame(predictive_rows)
predictive_metrics.to_csv(RESULTS_DIR/"predictive_metrics_separated_estimands.csv",index=False,encoding="utf-8-sig")
display(predictive_metrics.query("position=='ALL'"))


In [ ]:
common_sample_rows=[]

for season,d in [(2024,cal),(2025,test)]:

    dc=d[
        d["participated_next"].eq(1)
    ].copy()

    dc=dc.dropna(
        subset=[
            "historical_conditional_baseline",
            "pred_conditional_ensemble",
            "pontos_realizados",
        ]
    )

    for name,col in {
        "Historical conditional mean baseline":
            "historical_conditional_baseline",
        "OOF-weighted RF/XGB ensemble":
            "pred_conditional_ensemble",
    }.items():

        m=regression_metrics(
            dc["pontos_realizados"],
            dc[col]
        )

        common_sample_rows.append({
            "season":season,
            "estimand":
                "conditional_performance_given_participation",
            "sample":"common_baseline_vs_model",
            "model":name,
            **m,
        })

    du=d.dropna(
        subset=[
            "historical_unconditional_baseline",
            "pred_expected",
            "pontos_realizados",
        ]
    ).copy()

    for name,col in {
        "Historical unconditional mean baseline":
            "historical_unconditional_baseline",
        "Availability-adjusted expected prediction":
            "pred_expected",
    }.items():

        m=regression_metrics(
            du["pontos_realizados"],
            du[col]
        )

        common_sample_rows.append({
            "season":season,
            "estimand":"unconditional_realized_points",
            "sample":"common_baseline_vs_model",
            "model":name,
            **m,
        })

predictive_common_sample=pd.DataFrame(
    common_sample_rows
)

predictive_common_sample.to_csv(
    RESULTS_DIR / "predictive_metrics_common_sample.csv",
    index=False,
    encoding="utf-8-sig"
)

display(predictive_common_sample)


In [ ]:
availability_eval_rows=[]
availability_eval_rel=[]

for season,d in [(2024,cal),(2025,test)]:
    dd=d[
        d["posicao_id"].ne(6)
        & d["participated_next"].notna()
        & d["prob_play_estimated"].notna()
    ].copy()

    y=dd["participated_next"].astype(int).to_numpy()
    p=dd["prob_play_estimated"].to_numpy(float)

    intercept,slope,ece,rel=calibration_diagnostics(y,p)

    availability_eval_rows.append({
        "season":season,
        "n":len(dd),
        "log_loss":log_loss(y,p,labels=[0,1]),
        "brier":brier_score_loss(y,p),
        "roc_auc":roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan,
        "calibration_intercept":intercept,
        "calibration_slope":slope,
        "ece_10_bins":ece,
    })

    rel["season"]=season
    availability_eval_rel.append(rel)

availability_eval=pd.DataFrame(availability_eval_rows)
availability_reliability_eval=pd.concat(availability_eval_rel,ignore_index=True)

availability_eval.to_csv(RESULTS_DIR/"availability_probability_evaluation.csv",index=False,encoding="utf-8-sig")
availability_reliability_eval.to_csv(RESULTS_DIR/"availability_reliability_2024_2025.csv",index=False,encoding="utf-8-sig")

display(availability_eval)


In [ ]:
def percentile_component(s):
    s=pd.to_numeric(s,errors="coerce")
    return s.rank(method="average",pct=True,ascending=True)

def add_decision_components(df):
    d=df.copy()

    d["component_expected"]=d["pred_expected"]
    d["component_upside"]=d["upside_component_raw"]
    d["component_economic"]=d["economic_efficiency"]

    for c in ["expected","upside","economic"]:
        d[f"component_{c}_norm"]=(
            d.groupby(["temporada","rodada_target"])[f"component_{c}"]
            .transform(percentile_component)
        )

    return d

cal=add_decision_components(cal)
test=add_decision_components(test)

component_correlations=[]
for season,d in [(2024,cal),(2025,test)]:
    corr=d[
        ["component_expected_norm","component_upside_norm","component_economic_norm"]
    ].corr(method="spearman")

    for a,b in combinations(corr.columns,2):
        component_correlations.append({
            "season":season,
            "component_a":a,
            "component_b":b,
            "spearman":corr.loc[a,b],
        })

component_correlations=pd.DataFrame(component_correlations)
component_correlations.to_csv(RESULTS_DIR/"decision_component_correlations.csv",index=False,encoding="utf-8-sig")
display(component_correlations)


In [ ]:
def solve_formation(df_round,score_col,formation_name):
    d=df_round.copy()

    d=d[
        d["preco_num"].notna()
        & (d["preco_num"]>0)
        & d[score_col].notna()
        & d["pontos_realizados"].notna()
        & d["posicao_id"].notna()
    ].copy()

    req=FORMACOES[formation_name]

    for pos_name,qty in req.items():
        if qty<=0:
            continue
        pos_id=MAPA_POSICOES[pos_name]
        if (d["posicao_id"]==pos_id).sum()<qty:
            return None,f"insufficient_{pos_name}"

    problem=pulp.LpProblem(f"FAME_{formation_name}",pulp.LpMaximize)
    x={i:pulp.LpVariable(f"x_{i}",cat="Binary") for i in d.index}

    problem += pulp.lpSum(float(d.loc[i,score_col])*x[i] for i in d.index)
    problem += pulp.lpSum(float(d.loc[i,"preco_num"])*x[i] for i in d.index) <= BUDGET_MAX

    for pos_name,qty in req.items():
        pos_id=MAPA_POSICOES[pos_name]
        problem += pulp.lpSum(
            x[i] for i in d.index
            if int(d.loc[i,"posicao_id"])==pos_id
        ) == qty

    status=problem.solve(pulp.PULP_CBC_CMD(msg=False))

    if pulp.LpStatus[status]!="Optimal":
        return None,pulp.LpStatus[status]

    selected_idx=[
        i for i in d.index
        if x[i].value() is not None and x[i].value()>0.5
    ]

    team=d.loc[selected_idx].copy()
    team["formation_selected"]=formation_name

    return team,None

def select_best_lineup(df_round,score_col):
    candidates=[]
    failures=[]

    for formation in FORMACOES:
        team,reason=solve_formation(df_round,score_col,formation)

        if team is None:
            failures.append((formation,reason))
            continue

        objective=float(team[score_col].sum())
        cost=float(team["preco_num"].sum())

        candidates.append((objective,-cost,formation,team))

    if not candidates:
        return None,failures

    candidates.sort(
        key=lambda z:(z[0],z[1],z[2]),
        reverse=True
    )

    return candidates[0][3],failures

def lineup_signature(team):
    return "|".join(sorted(team["atleta_id"].astype(str).tolist()))

def evaluate_score(df,score_col,strategy,metadata=None):
    metadata=metadata or {}
    round_rows=[]
    player_parts=[]
    failures=[]

    for (season,r),g in df.groupby(["temporada","rodada_target"]):
        team,failure_info=select_best_lineup(g,score_col)

        if team is None:
            failures.append({
                "temporada":season,
                "rodada_target":r,
                "strategy":strategy,
                "reason":str(failure_info),
                **metadata,
            })
            continue

        utility=float(team["pontos_realizados"].sum())
        cost=float(team["preco_num"].sum())

        round_rows.append({
            "temporada":season,
            "rodada_target":r,
            "strategy":strategy,
            "score_column":score_col,
            "operational_utility_no_captain":utility,
            "objective_value":float(team[score_col].sum()),
            "cost_total":cost,
            "budget_remaining":BUDGET_MAX-cost,
            "budget_binding_indicator":bool(abs(BUDGET_MAX-cost)<1e-6),
            "formation_selected":team["formation_selected"].iloc[0],
            "n_selected":len(team),
            "lineup_signature":lineup_signature(team),
            **metadata,
        })

        det=team.copy()
        det["strategy"]=strategy
        det["temporada_backtest"]=season
        det["rodada_backtest"]=r

        for k,v in metadata.items():
            det[k]=v

        player_parts.append(det)

    return (
        pd.DataFrame(round_rows),
        pd.concat(player_parts,ignore_index=True) if player_parts else pd.DataFrame(),
        pd.DataFrame(failures),
    )


In [ ]:
def generate_weight_grid(step=WEIGHT_GRID_STEP):
    n=int(round(1/step))
    rows=[]
    for a in range(n+1):
        for b in range(n+1-a):
            c=n-a-b
            rows.append((a/n,b/n,c/n))
    return rows

weight_grid=generate_weight_grid()
weight_cols=["w_expected","w_upside","w_economic"]

pd.DataFrame(
    weight_grid,
    columns=weight_cols
).to_csv(DECISION_DIR/"doc_weight_grid.csv",index=False,encoding="utf-8-sig")

cal_round_parts=[]
cal_player_parts=[]

for wp,wu,we in weight_grid:
    d=cal.copy()

    d["_score_doc_candidate"]=(
        wp*d["component_expected_norm"]
        + wu*d["component_upside_norm"]
        + we*d["component_economic_norm"]
    )

    rr,pp,_=evaluate_score(
        d,
        "_score_doc_candidate",
        "DOC calibration candidate",
        {
            "w_expected":wp,
            "w_upside":wu,
            "w_economic":we,
        }
    )

    cal_round_parts.append(rr)
    cal_player_parts.append(pp)

cal_results_round=pd.concat(cal_round_parts,ignore_index=True)
cal_results_players=pd.concat(cal_player_parts,ignore_index=True)

cal_summary=(
    cal_results_round.groupby(weight_cols,as_index=False)
    .agg(
        n_rounds=("operational_utility_no_captain","size"),
        mean_utility=("operational_utility_no_captain","mean"),
        median_utility=("operational_utility_no_captain","median"),
        sd_utility=("operational_utility_no_captain","std"),
        cumulative_utility=("operational_utility_no_captain","sum"),
        mean_cost=("cost_total","mean"),
        pct_budget_binding=("budget_binding_indicator","mean"),
    )
    .sort_values(["mean_utility","sd_utility"],ascending=[False,True])
    .reset_index(drop=True)
)

cal_summary["rank_mean"]=np.arange(1,len(cal_summary)+1)
best=cal_summary.iloc[0]

DOC_WEIGHTS=(
    float(best["w_expected"]),
    float(best["w_upside"]),
    float(best["w_economic"]),
)

best_mean=float(best["mean_utility"])
cal_summary["within_1pct_best"]=(
    cal_summary["mean_utility"] >= (1-NEAR_OPTIMAL_TOLERANCE)*best_mean
)

cal_results_round.to_csv(DECISION_DIR/"doc_calibration_by_round_all_weights.csv",index=False,encoding="utf-8-sig")
cal_results_players.to_csv(DECISION_DIR/"doc_calibration_selected_players_all_weights.csv",index=False,encoding="utf-8-sig")
cal_summary.to_csv(DECISION_DIR/"doc_calibration_summary.csv",index=False,encoding="utf-8-sig")

pd.DataFrame([{
    "w_expected":DOC_WEIGHTS[0],
    "w_upside":DOC_WEIGHTS[1],
    "w_economic":DOC_WEIGHTS[2],
}]).to_csv(DECISION_DIR/"frozen_doc_weights.csv",index=False,encoding="utf-8-sig")

print("DOC weights:",DOC_WEIGHTS)
display(cal_summary.head(15))


In [ ]:
cal_tmp=cal_results_round.copy()
cal_tmp["weight_id"]=cal_tmp[weight_cols].round(8).astype(str).agg("|".join,axis=1)

pivot_cal=cal_tmp.pivot_table(
    index=["temporada","rodada_target"],
    columns="weight_id",
    values="operational_utility_no_captain",
    aggfunc="first"
).dropna(axis=0,how="any")

meta_weights=(
    cal_tmp[["weight_id"]+weight_cols]
    .drop_duplicates("weight_id")
    .set_index("weight_id")
    .loc[pivot_cal.columns]
)

M=pivot_cal.to_numpy(float)
n_rounds=M.shape[0]

selection_count={wid:0 for wid in pivot_cal.columns}

rg=np.random.default_rng(BOOTSTRAP_SEED)

for _ in range(N_BOOTSTRAP):
    idx=rg.integers(0,n_rounds,size=n_rounds)
    means=M[idx,:].mean(axis=0)
    selection_count[pivot_cal.columns[int(np.argmax(means))]] += 1

rows=[]
for wid,count in selection_count.items():
    m=meta_weights.loc[wid]
    rows.append({
        "weight_id":wid,
        **{c:float(m[c]) for c in weight_cols},
        "selection_count":int(count),
        "selection_frequency_pct":100*count/N_BOOTSTRAP,
    })

cal_selection_stability=(
    pd.DataFrame(rows)
    .sort_values("selection_frequency_pct",ascending=False)
    .reset_index(drop=True)
)

cal_selection_stability.to_csv(
    DECISION_DIR/"doc_calibration_bootstrap_stability.csv",
    index=False,
    encoding="utf-8-sig"
)

display(cal_selection_stability.head(15))


In [ ]:
def add_weighted_score(d,weights,col):
    wp,wu,we=weights

    d[col]=(
        wp*d["component_expected_norm"]
        + wu*d["component_upside_norm"]
        + we*d["component_economic_norm"]
    )

    return d

test_eval=test.copy()

test_eval=add_weighted_score(
    test_eval,
    DOC_WEIGHTS,
    "score_fame_doc"
)

test_eval=add_weighted_score(
    test_eval,
    FAME_ORIGINAL_WEIGHTS,
    "score_fame_original"
)

test_eval=add_weighted_score(
    test_eval,
    EQUAL_WEIGHTS,
    "score_equal_weight"
)

test_eval["score_expected_only_normalized"] = (
    test_eval["component_expected_norm"]
)

strategy_cols={
    "FAME-DOC":"score_fame_doc",
    "Expected-only normalized representation":
        "score_expected_only_normalized",
    "Availability-adjusted expected prediction":
        "pred_expected",
    "FAME-Original 70/20/10":
        "score_fame_original",
    "Equal-weight representation":
        "score_equal_weight",
    "Pure conditional prediction":
        "pred_conditional_ensemble",
    "Upside/dispersion only":
        "upside_component_raw",
    "Economic efficiency":
        "economic_efficiency",
    "Historical mean":
        "media_num",
    "Market value":
        "preco_num",
}

test_round_parts=[]
test_player_parts=[]
test_fail_parts=[]

for strategy,col in strategy_cols.items():

    rr,pp,ff=evaluate_score(
        test_eval,
        col,
        strategy
    )

    test_round_parts.append(rr)
    test_player_parts.append(pp)

    if not ff.empty:
        test_fail_parts.append(ff)

test_round=pd.concat(
    test_round_parts,
    ignore_index=True
)

test_players=pd.concat(
    test_player_parts,
    ignore_index=True
)

test_failures=(
    pd.concat(
        test_fail_parts,
        ignore_index=True
    )
    if test_fail_parts
    else pd.DataFrame()
)

test_summary=(
    test_round
    .groupby("strategy",as_index=False)
    .agg(
        n_rounds=("operational_utility_no_captain","size"),
        mean_utility=("operational_utility_no_captain","mean"),
        median_utility=("operational_utility_no_captain","median"),
        sd_utility=("operational_utility_no_captain","std"),
        cumulative_utility=("operational_utility_no_captain","sum"),
        mean_cost=("cost_total","mean"),
        min_budget_remaining=("budget_remaining","min"),
        pct_budget_binding=("budget_binding_indicator","mean"),
    )
    .sort_values("mean_utility",ascending=False)
)

test_round.to_csv(
    RESULTS_DIR / "operational_utility_by_round_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

test_players.to_csv(
    RESULTS_DIR / "selected_players_all_strategies_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

test_summary.to_csv(
    RESULTS_DIR / "operational_summary_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

display(test_summary)


In [ ]:
wide=test_round.pivot_table(
    index=["temporada","rodada_target"],
    columns="strategy",
    values="operational_utility_no_captain",
    aggfunc="first"
).sort_index()

PRIMARY_A="FAME-DOC"
PRIMARY_B="Expected-only normalized representation"

primary=wide[[PRIMARY_A,PRIMARY_B]].dropna()
delta=(primary[PRIMARY_A]-primary[PRIMARY_B]).to_numpy(float)

def paired_bootstrap_ci(delta,n_boot=N_BOOTSTRAP,seed=BOOTSTRAP_SEED):
    delta=np.asarray(delta,float)
    rg=np.random.default_rng(seed)
    means=rg.choice(
        delta,
        size=(n_boot,len(delta)),
        replace=True
    ).mean(axis=1)

    lo,hi=np.quantile(means,[0.025,0.975])
    return float(lo),float(hi)

def circular_moving_block_bootstrap_ci(
    delta,
    block_length,
    n_boot=N_BOOTSTRAP,
    seed=BOOTSTRAP_SEED
):
    x=np.asarray(delta,float)
    n=len(x)
    rg=np.random.default_rng(seed+block_length)

    means=[]

    for _ in range(n_boot):
        sample=[]

        while len(sample)<n:
            start=int(rg.integers(0,n))
            block=[
                x[(start+j)%n]
                for j in range(block_length)
            ]
            sample.extend(block)

        means.append(np.mean(sample[:n]))

    lo,hi=np.quantile(means,[0.025,0.975])
    return float(lo),float(hi)

lo,hi=paired_bootstrap_ci(delta)

try:
    wstat,wp=wilcoxon(
        delta,
        zero_method="wilcox",
        alternative="two-sided"
    )
except ValueError:
    wstat,wp=0.0,1.0

primary_rows=[{
    "contrast":f"{PRIMARY_A} minus {PRIMARY_B}",
    "n_rounds":len(delta),
    "mean_difference":float(np.mean(delta)),
    "median_difference":float(np.median(delta)),
    "ci_method":"paired_iid_bootstrap",
    "block_length":np.nan,
    "ci95_low":lo,
    "ci95_high":hi,
    "wilcoxon_stat":wstat,
    "wilcoxon_p_unadjusted_scientific_contrast":wp,
}]

for L in [2,3,4,5,6]:
    blo,bhi=circular_moving_block_bootstrap_ci(delta,L)
    primary_rows.append({
        "contrast":f"{PRIMARY_A} minus {PRIMARY_B}",
        "n_rounds":len(delta),
        "mean_difference":float(np.mean(delta)),
        "median_difference":float(np.median(delta)),
        "ci_method":"circular_moving_block_bootstrap",
        "block_length":L,
        "ci95_low":blo,
        "ci95_high":bhi,
        "wilcoxon_stat":np.nan,
        "wilcoxon_p_unadjusted_scientific_contrast":np.nan,
    })

primary_inference=pd.DataFrame(primary_rows)
primary_inference.to_csv(
    RESULTS_DIR/"primary_doc_vs_expected_normalized_inference.csv",
    index=False,
    encoding="utf-8-sig"
)

secondary=wide[["FAME-DOC","FAME-Original 70/20/10"]].dropna()
delta_orig=(secondary["FAME-DOC"]-secondary["FAME-Original 70/20/10"]).to_numpy(float)
olo,ohi=paired_bootstrap_ci(delta_orig,seed=BOOTSTRAP_SEED+77)

secondary_historical=pd.DataFrame([{
    "contrast":"FAME-DOC minus FAME-Original 70/20/10",
    "n_rounds":len(delta_orig),
    "mean_difference":delta_orig.mean(),
    "ci95_low":olo,
    "ci95_high":ohi,
}])

secondary_historical.to_csv(
    RESULTS_DIR/"secondary_doc_vs_original_paired.csv",
    index=False,
    encoding="utf-8-sig"
)

display(primary_inference)


In [ ]:
strategies_complete=[
    c for c in wide.columns
    if wide[c].notna().all()
]

fstat,fp=friedmanchisquare(
    *[wide[c].to_numpy() for c in strategies_complete]
)

friedman_table=pd.DataFrame([{
    "n_strategies":len(strategies_complete),
    "n_rounds":len(wide),
    "friedman_stat":fstat,
    "p_value":fp,
}])

def holm_adjust(p):
    p=np.asarray(p,float)
    order=np.argsort(p)
    out=np.empty_like(p)
    running=0.0
    m=len(p)

    for rank,idx in enumerate(order):
        val=min(1.0,(m-rank)*p[idx])
        running=max(running,val)
        out[idx]=running

    return out

pairs=[]

for a,b in combinations(strategies_complete,2):
    dd=(wide[a]-wide[b]).dropna()

    try:
        stat,pv=wilcoxon(
            dd,
            zero_method="wilcox",
            alternative="two-sided"
        )
    except ValueError:
        stat,pv=0.0,1.0

    pairs.append({
        "strategy_a":a,
        "strategy_b":b,
        "mean_diff_a_minus_b":dd.mean(),
        "wilcoxon_stat":stat,
        "p_raw":pv,
    })

pairwise=pd.DataFrame(pairs)
pairwise["p_holm"]=holm_adjust(pairwise["p_raw"].to_numpy())

friedman_table.to_csv(RESULTS_DIR/"friedman_all_strategies.csv",index=False,encoding="utf-8-sig")
pairwise.to_csv(RESULTS_DIR/"pairwise_wilcoxon_holm.csv",index=False,encoding="utf-8-sig")

display(friedman_table)


In [ ]:
def percentile_bootstrap_ci(x,n_boot=N_BOOTSTRAP,seed=BOOTSTRAP_SEED):
    x=np.asarray(pd.Series(x).dropna(),dtype=float)
    rg=np.random.default_rng(seed)

    means=rg.choice(
        x,
        size=(n_boot,len(x)),
        replace=True
    ).mean(axis=1)

    return np.quantile(means,[0.025,0.975])

rows=[]

for i,(strategy,g) in enumerate(test_round.groupby("strategy")):
    lo,hi=percentile_bootstrap_ci(
        g["operational_utility_no_captain"],
        seed=BOOTSTRAP_SEED+i
    )

    rows.append({
        "strategy":strategy,
        "mean_utility":g["operational_utility_no_captain"].mean(),
        "ci95_low":lo,
        "ci95_high":hi,
    })

bootstrap_table=pd.DataFrame(rows)
bootstrap_table.to_csv(RESULTS_DIR/"strategy_bootstrap_ci_2025.csv",index=False,encoding="utf-8-sig")


In [ ]:
test_observed_only=test_eval[test_eval["has_next_market_row"].eq(True)].copy()

sens_round_parts=[]

for strategy,col in {
    "FAME-DOC":"score_fame_doc",
    "Availability-adjusted expected prediction":"pred_expected",
}.items():
    rr,_,_=evaluate_score(
        test_observed_only,
        col,
        strategy
    )
    rr["missing_next_market_scenario"]="exclude_missing_next_market"
    sens_round_parts.append(rr)

sens_exclude=pd.concat(sens_round_parts,ignore_index=True)

sens_main=test_round[
    test_round["strategy"].isin(
        ["FAME-DOC","Availability-adjusted expected prediction"]
    )
].copy()
sens_main["missing_next_market_scenario"]="missing_next_market_as_zero"

missing_market_sensitivity=pd.concat(
    [sens_main,sens_exclude],
    ignore_index=True
)

missing_market_summary=(
    missing_market_sensitivity.groupby(
        ["missing_next_market_scenario","strategy"],
        as_index=False
    )
    .agg(
        n_rounds=("operational_utility_no_captain","size"),
        mean_utility=("operational_utility_no_captain","mean"),
    )
)

missing_market_sensitivity.to_csv(
    RESULTS_DIR/"missing_next_market_sensitivity_by_round.csv",
    index=False,
    encoding="utf-8-sig"
)
missing_market_summary.to_csv(
    RESULTS_DIR/"missing_next_market_sensitivity_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

display(missing_market_summary)


In [ ]:
def _dcg(rel):
    rel=np.asarray(rel,dtype=float)

    if rel.size==0:
        return np.nan

    return float(
        np.sum(rel/np.log2(np.arange(2,len(rel)+2)))
    )

def ranking_metrics_round(group,criterion,k_requested):
    d=group.dropna(
        subset=[criterion,"pontos_realizados"]
    ).drop_duplicates("atleta_id").copy()

    if len(d)<2:
        return None

    pred_rank=d.sort_values(
        [criterion,"preco_num","atleta_id"],
        ascending=[False,True,True]
    )

    real_rank=d.sort_values(
        ["pontos_realizados","atleta_id"],
        ascending=[False,True]
    )

    k=min(k_requested,len(d))
    ids_pred=pred_rank.head(k)["atleta_id"].tolist()
    ids_real=set(real_rank.head(k)["atleta_id"].tolist())

    overlap=len(set(ids_pred)&ids_real)

    rp=d[criterion].rank(ascending=False)
    rr=d["pontos_realizados"].rank(ascending=False)

    rel=d.set_index("atleta_id")["pontos_realizados"]
    shift=max(0.0,-float(rel.min()))
    rel=(rel+shift).clip(lower=0)

    dcg=_dcg(rel.reindex(ids_pred).fillna(0).to_numpy())
    idcg=_dcg(np.sort(rel.to_numpy())[::-1][:k])

    return {
        "precision_at_k":overlap/k,
        "recall_at_k":overlap/len(ids_real) if ids_real else np.nan,
        "ndcg_at_k":dcg/idcg if idcg and idcg>0 else np.nan,
        "spearman":spearmanr(rp,rr,nan_policy="omit").statistic,
        "kendall":kendalltau(rp,rr,nan_policy="omit").statistic,
    }

rank_rows=[]

for strategy,col in strategy_cols.items():
    for (season,r),g in test_eval.groupby(["temporada","rodada_target"]):
        for k in [5,10,20]:
            x=ranking_metrics_round(g,col,k)

            if x is not None:
                rank_rows.append({
                    "strategy":strategy,
                    "criterion":col,
                    "temporada":season,
                    "rodada_target":r,
                    "k":k,
                    **x,
                })

ranking_round=pd.DataFrame(rank_rows)

ranking_summary=(
    ranking_round.groupby(
        ["strategy","criterion","k"],
        as_index=False
    )
    .agg(
        precision_mean=("precision_at_k","mean"),
        recall_mean=("recall_at_k","mean"),
        ndcg_mean=("ndcg_at_k","mean"),
        spearman_mean=("spearman","mean"),
        kendall_mean=("kendall","mean"),
    )
)

ranking_round.to_csv(RESULTS_DIR/"ranking_metrics_by_round.csv",index=False,encoding="utf-8-sig")
ranking_summary.to_csv(RESULTS_DIR/"ranking_metrics_summary.csv",index=False,encoding="utf-8-sig")


In [ ]:
posthoc_parts=[]

for wp,wu,we in weight_grid:
    d=test.copy()

    d["_score_posthoc"]=(
        wp*d["component_expected_norm"]
        + wu*d["component_upside_norm"]
        + we*d["component_economic_norm"]
    )

    rr,_,_=evaluate_score(
        d,
        "_score_posthoc",
        "2025 post-hoc",
        {
            "w_expected":wp,
            "w_upside":wu,
            "w_economic":we,
        }
    )

    posthoc_parts.append(rr)

posthoc_round=pd.concat(posthoc_parts,ignore_index=True)

posthoc_summary=(
    posthoc_round.groupby(weight_cols,as_index=False)
    .agg(
        mean_utility=("operational_utility_no_captain","mean"),
        sd_utility=("operational_utility_no_captain","std"),
    )
    .sort_values(["mean_utility","sd_utility"],ascending=[False,True])
    .reset_index(drop=True)
)

posthoc_summary["rank_mean"]=np.arange(1,len(posthoc_summary)+1)

RETROSPECTIVE_BEST_2025=tuple(
    posthoc_summary.iloc[0][weight_cols].astype(float).tolist()
)

posthoc_round.to_csv(
    RESULTS_DIR/"posthoc_2025_all_weights_by_round.csv",
    index=False,
    encoding="utf-8-sig"
)

posthoc_summary.to_csv(
    RESULTS_DIR/"posthoc_2025_weight_landscape.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Retrospective best 2025 (diagnostic only):",RETROSPECTIVE_BEST_2025)


In [ ]:
def jaccard_signature(a,b):
    A=set(str(a).split("|"))
    B=set(str(b).split("|"))
    return len(A&B)/len(A|B) if A|B else np.nan

lineup_wide=test_round.pivot_table(
    index=["temporada","rodada_target"],
    columns="strategy",
    values="lineup_signature",
    aggfunc="first"
).reset_index()

similarity_rows=[]

comparators=[
    "Expected-only normalized representation",
    "Availability-adjusted expected prediction",
    "FAME-Original 70/20/10",
]

for comparator in comparators:
    if {"FAME-DOC",comparator}.issubset(lineup_wide.columns):
        for _,r in lineup_wide.iterrows():
            similarity_rows.append({
                "temporada":r["temporada"],
                "rodada_target":r["rodada_target"],
                "comparator":comparator,
                "jaccard":jaccard_signature(
                    r["FAME-DOC"],
                    r[comparator]
                ),
                "identical":r["FAME-DOC"]==r[comparator],
            })

lineup_similarity=pd.DataFrame(similarity_rows)
lineup_similarity.to_csv(
    RESULTS_DIR/"lineup_similarity_doc_comparators.csv",
    index=False,
    encoding="utf-8-sig"
)

utility_wide=wide.reset_index()

for comparator in [
    "Expected-only normalized representation",
    "Availability-adjusted expected prediction",
    "FAME-Original 70/20/10",
    "Equal-weight representation",
]:
    if comparator in utility_wide.columns:
        d=utility_wide[
            ["temporada","rodada_target","FAME-DOC",comparator]
        ].copy()

        d["delta_doc_minus_comparator"]=(
            d["FAME-DOC"]-d[comparator]
        )

        safe=re.sub(
            r"[^a-z0-9]+",
            "_",
            comparator.lower()
        ).strip("_")

        d.to_csv(
            RESULTS_DIR/f"roundwise_delta_doc_vs_{safe}.csv",
            index=False,
            encoding="utf-8-sig"
        )


In [ ]:
orig=np.asarray(FAME_ORIGINAL_WEIGHTS,float)

grid_local=pd.DataFrame(
    weight_grid,
    columns=weight_cols
)

grid_local["l1_distance_to_original"]=grid_local[weight_cols].apply(
    lambda r:float(np.abs(r.to_numpy(float)-orig).sum()),
    axis=1
)

local_results=(
    grid_local
    .merge(
        cal_summary[weight_cols+["mean_utility","rank_mean"]].rename(
            columns={
                "mean_utility":"mean_utility_2024",
                "rank_mean":"rank_2024"
            }
        ),
        on=weight_cols,
        how="left"
    )
    .merge(
        posthoc_summary[weight_cols+["mean_utility","rank_mean"]].rename(
            columns={
                "mean_utility":"mean_utility_2025",
                "rank_mean":"rank_2025"
            }
        ),
        on=weight_cols,
        how="left"
    )
)

local_summary=[]

for radius in [0.10,0.20]:
    d=local_results[
        local_results["l1_distance_to_original"]<=radius+1e-12
    ].copy()

    local_summary.append({
        "l1_radius":radius,
        "n_vectors":len(d),
        "mean_region_2024":d["mean_utility_2024"].mean(),
        "mean_region_2025":d["mean_utility_2025"].mean(),
        "min_region_2025":d["mean_utility_2025"].min(),
        "max_region_2025":d["mean_utility_2025"].max(),
    })

local_sensitivity_summary=pd.DataFrame(local_summary)

local_results.to_csv(
    RESULTS_DIR/"original_reference_local_sensitivity.csv",
    index=False,
    encoding="utf-8-sig"
)

local_sensitivity_summary.to_csv(
    RESULTS_DIR/"original_reference_local_sensitivity_summary.csv",
    index=False,
    encoding="utf-8-sig"
)


In [ ]:
doc_w=np.asarray(DOC_WEIGHTS,float)
ab=test.copy()

def score_from_components(d,weights,active,col):
    w=np.asarray(weights,float).copy()
    mask=np.asarray(active,bool)
    w[~mask]=0

    if w.sum()==0:
        raise ValueError("Nenhum peso ativo.")

    w=w/w.sum()

    d[col]=(
        w[0]*d["component_expected_norm"]
        + w[1]*d["component_upside_norm"]
        + w[2]*d["component_economic_norm"]
    )

    return d

ab=score_from_components(ab,doc_w,[True,True,True],"ab_full_doc")
ab=score_from_components(ab,doc_w,[True,False,True],"ab_no_upside")
ab=score_from_components(ab,doc_w,[True,True,False],"ab_no_economic")

ab["na_expected"]=ab["pred_conditional_ensemble"]
ab["na_upside"]=ab["historical_uncertainty"]
ab["na_economic"]=(
    ab["pred_conditional_ensemble"]
    / ab["preco_num"].where(ab["preco_num"]>0)
)

for c in ["expected","upside","economic"]:
    ab[f"na_{c}_norm"]=(
        ab.groupby(["temporada","rodada_target"])[f"na_{c}"]
        .transform(percentile_component)
    )

ab["ab_no_availability"]=(
    doc_w[0]*ab["na_expected_norm"]
    + doc_w[1]*ab["na_upside_norm"]
    + doc_w[2]*ab["na_economic_norm"]
)

ab["ab_expected_only_normalized"]=ab["component_expected_norm"]
ab["ab_expected_prediction"]=ab["pred_expected"]
ab["ab_conditional_prediction"]=ab["pred_conditional_ensemble"]
ab["ab_equal_weight"]=(
    ab["component_expected_norm"]
    + ab["component_upside_norm"]
    + ab["component_economic_norm"]
)/3

ABLATIONS={
    "Full FAME-DOC":"ab_full_doc",
    "No upside":"ab_no_upside",
    "No economic efficiency":"ab_no_economic",
    "No availability adjustment":"ab_no_availability",
    "Expected-only normalized representation":"ab_expected_only_normalized",
    "Availability-adjusted expected prediction":"ab_expected_prediction",
    "Pure conditional prediction":"ab_conditional_prediction",
    "Equal-weight representation":"ab_equal_weight",
}

ab_round_parts=[]
ab_player_parts=[]

for name,col in ABLATIONS.items():
    rr,pp,_=evaluate_score(ab,col,name)
    ab_round_parts.append(rr)
    ab_player_parts.append(pp)

ablation_round=pd.concat(ab_round_parts,ignore_index=True)
ablation_players=pd.concat(ab_player_parts,ignore_index=True)

ablation_summary=(
    ablation_round.groupby("strategy",as_index=False)
    .agg(
        mean_utility=("operational_utility_no_captain","mean"),
        median_utility=("operational_utility_no_captain","median"),
        sd_utility=("operational_utility_no_captain","std"),
        cumulative_utility=("operational_utility_no_captain","sum"),
    )
    .sort_values("mean_utility",ascending=False)
)

ablation_round.to_csv(RESULTS_DIR/"ablation_by_round_2025.csv",index=False,encoding="utf-8-sig")
ablation_players.to_csv(RESULTS_DIR/"ablation_selected_players_2025.csv",index=False,encoding="utf-8-sig")
ablation_summary.to_csv(RESULTS_DIR/"ablation_summary_2025.csv",index=False,encoding="utf-8-sig")

display(ablation_summary)


In [ ]:
CAPTAIN_BONUS_RATE=0.50

CAPTAIN_LINEUP_STRATEGIES=[
    "FAME-DOC",
    "Availability-adjusted expected prediction",
    "FAME-Original 70/20/10",
]

captain_rules={
    "Highest Market Value":"preco_num",
    "Highest Historical Mean":"media_num",
    "Highest Expected Performance":"pred_expected",
    "Highest Conditional Prediction":"pred_conditional_ensemble",
}

captain_rows=[]

for lineup_strategy in CAPTAIN_LINEUP_STRATEGIES:
    selected=test_players[
        test_players["strategy"].eq(lineup_strategy)
    ].copy()

    for (season,r),g0 in selected.groupby(
        ["temporada_backtest","rodada_backtest"]
    ):
        g=g0[g0["posicao_id"].ne(6)].copy()

        if len(g)!=11:
            continue

        complete_utility=float(
            test_round.loc[
                test_round["strategy"].eq(lineup_strategy)
                & test_round["temporada"].eq(season)
                & test_round["rodada_target"].eq(r),
                "operational_utility_no_captain"
            ].iloc[0]
        )

        oracle=g.sort_values(
            ["pontos_realizados","atleta_id"],
            ascending=[False,True]
        ).iloc[0]

        rules=dict(captain_rules)

        if lineup_strategy=="FAME-DOC":
            rules["Highest FAME Representation"]="score_fame_doc"
        elif lineup_strategy=="FAME-Original 70/20/10":
            rules["Highest FAME Representation"]="score_fame_original"

        for rule,col in rules.items():
            cand=g.dropna(subset=[col]).sort_values(
                [col,"preco_num","atleta_id"],
                ascending=[False,True,True]
            )

            if cand.empty:
                continue

            cap=cand.iloc[0]
            bonus=CAPTAIN_BONUS_RATE*float(cap["pontos_realizados"])

            captain_rows.append({
                "temporada":season,
                "rodada_target":r,
                "lineup_strategy":lineup_strategy,
                "captain_rule":rule,
                "captain_player_id":cap["atleta_id"],
                "captain_realized_score":float(cap["pontos_realizados"]),
                "captain_bonus":bonus,
                "final_team_utility_with_captain":complete_utility+bonus,
                "oracle_realized_score":float(oracle["pontos_realizados"]),
                "captain_regret_score":float(
                    oracle["pontos_realizados"]
                    - cap["pontos_realizados"]
                ),
                "is_oracle":bool(
                    cap["atleta_id"]==oracle["atleta_id"]
                ),
            })

captain_results=pd.DataFrame(captain_rows)

captain_summary=(
    captain_results.groupby(
        ["lineup_strategy","captain_rule"],
        as_index=False
    )
    .agg(
        mean_captain_bonus=("captain_bonus","mean"),
        mean_final_team_utility=("final_team_utility_with_captain","mean"),
        mean_regret=("captain_regret_score","mean"),
        oracle_hit_rate=("is_oracle","mean"),
    )
    .sort_values(
        ["lineup_strategy","mean_final_team_utility"],
        ascending=[True,False]
    )
)

captain_results.to_csv(RESULTS_DIR/"captain_results_by_round.csv",index=False,encoding="utf-8-sig")
captain_summary.to_csv(RESULTS_DIR/"captain_summary.csv",index=False,encoding="utf-8-sig")

display(captain_summary)


In [ ]:
def simplex_xy(a,b,c):
    return b+0.5*c,(np.sqrt(3)/2)*c

def savefig(stem):
    plt.tight_layout()
    plt.savefig(FIG_DIR/f"{stem}.png",dpi=300,bbox_inches="tight")
    plt.savefig(FIG_DIR/f"{stem}.pdf",bbox_inches="tight")
    plt.show()
    plt.close()

best_tuning=(
    tuning_summary.sort_values(["position","model","mean_RMSE"])
    .groupby(["position","model"],as_index=False)
    .first()
)

plt.figure(figsize=(9,5.5))
for model,g in best_tuning.groupby("model"):
    plt.plot(g["position"],g["mean_RMSE"],marker="o",label=model)
plt.ylabel("Mean temporal-CV RMSE")
plt.xlabel("")
plt.title("Selected predictive configurations across tactical positions")
plt.legend(frameon=False)
savefig("fig_predictive_tuning_temporal_cv")

plt.figure(figsize=(6.5,6.5))
for season,g in availability_reliability_eval.groupby("season"):
    plt.plot(
        g["mean_predicted_probability"],
        g["observed_frequency"],
        marker="o",
        label=str(season)
    )
plt.plot([0,1],[0,1],linestyle="--")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed participation frequency")
plt.title("Availability probability calibration")
plt.legend(frameon=False)
savefig("fig_availability_calibration")

cp=cal_summary.copy()
xy=cp.apply(
    lambda r:simplex_xy(
        r["w_expected"],
        r["w_upside"],
        r["w_economic"]
    ),
    axis=1
)

cp["x"]=[z[0] for z in xy]
cp["y"]=[z[1] for z in xy]

plt.figure(figsize=(8.2,6.8))
sc=plt.scatter(cp["x"],cp["y"],c=cp["mean_utility"],s=48)
plt.colorbar(sc,label="Mean realized utility (2024)")
docrow=cp.iloc[0]
plt.scatter([docrow["x"]],[docrow["y"]],marker="*",s=220,label="FAME-DOC")
plt.plot([0,1,.5,0],[0,0,np.sqrt(3)/2,0],linewidth=1)
plt.xticks([])
plt.yticks([])
plt.legend(frameon=False)
plt.title("Decision-layer calibration landscape in 2024")
savefig("fig_2024_doc_calibration_landscape")

perf=(
    test_summary.merge(
        bootstrap_table,
        on=["strategy","mean_utility"],
        how="left"
    )
    .sort_values("mean_utility")
)

xerr=np.vstack([
    perf["mean_utility"].to_numpy()-perf["ci95_low"].to_numpy(),
    perf["ci95_high"].to_numpy()-perf["mean_utility"].to_numpy(),
])

plt.figure(figsize=(9,6.5))
plt.errorbar(
    perf["mean_utility"],
    perf["strategy"],
    xerr=xerr,
    fmt="o",
    capsize=3
)
plt.xlabel("Mean realized lineup utility in 2025")
plt.ylabel("")
plt.title("Independent out-of-time operational performance")
savefig("fig_2025_operational_utility_ci")

st=cal_selection_stability.head(15).copy()
st["label"]=st.apply(
    lambda r:f"({r.w_expected:.2f},{r.w_upside:.2f},{r.w_economic:.2f})",
    axis=1
)
st=st.sort_values("selection_frequency_pct")

plt.figure(figsize=(9,6))
plt.barh(st["label"],st["selection_frequency_pct"])
plt.xlabel("Bootstrap selection frequency (%)")
plt.ylabel("Weight vector")
plt.title("Stability of the 2024 DOC calibration optimum")
savefig("fig_2024_doc_selection_stability")

pp=posthoc_summary.copy()
xy=pp.apply(
    lambda r:simplex_xy(
        r["w_expected"],
        r["w_upside"],
        r["w_economic"]
    ),
    axis=1
)

pp["x"]=[z[0] for z in xy]
pp["y"]=[z[1] for z in xy]

plt.figure(figsize=(8.2,6.8))
sc=plt.scatter(pp["x"],pp["y"],c=pp["mean_utility"],s=48)
plt.colorbar(sc,label="Mean realized utility (2025, post hoc)")
best25=pp.iloc[0]
plt.scatter(
    [best25["x"]],
    [best25["y"]],
    marker="*",
    s=220,
    label="Retrospective best 2025"
)
plt.plot([0,1,.5,0],[0,0,np.sqrt(3)/2,0],linewidth=1)
plt.xticks([])
plt.yticks([])
plt.legend(frameon=False)
plt.title("Post-hoc decision-layer landscape in 2025")
savefig("fig_2025_posthoc_landscape")

aa=ablation_summary.sort_values("mean_utility")
plt.figure(figsize=(9,5.5))
plt.barh(aa["strategy"],aa["mean_utility"])
plt.xlabel("Mean realized utility in 2025")
plt.ylabel("")
plt.title("Ablation study")
savefig("fig_2025_ablation")

cc=captain_summary.sort_values(
    ["lineup_strategy","mean_final_team_utility"]
)

plt.figure(figsize=(10,6))
for strat,g in cc.groupby("lineup_strategy"):
    plt.plot(
        g["mean_final_team_utility"],
        g["captain_rule"],
        marker="o",
        label=strat
    )
plt.xlabel("Mean final team utility with captain")
plt.ylabel("")
plt.title("Secondary captain-selection analysis")
plt.legend(frameon=False)
savefig("fig_2025_captain_selection")


In [ ]:
budget_formation_audit=(
    test_round.groupby(
        ["strategy","formation_selected"],
        as_index=False
    )
    .agg(
        n_rounds=("rodada_target","size"),
        mean_cost=("cost_total","mean"),
        min_budget_remaining=("budget_remaining","min"),
        pct_binding=("budget_binding_indicator","mean"),
    )
)

budget_formation_audit.to_csv(
    AUDIT_DIR/"budget_and_formation_audit.csv",
    index=False,
    encoding="utf-8-sig"
)

display(budget_formation_audit)


In [ ]:
protocol=pd.DataFrame([
    {
        "item":"development/tuning period",
        "value":"2021-2023 expanding-window temporal validation",
        "uses_2025":False,
    },
    {
        "item":"availability regularization",
        "value":BEST_AVAILABILITY_C,
        "uses_2025":False,
    },
    {
        "item":"predictive hyperparameters",
        "value":"Sobol temporal CV 2021-2023",
        "uses_2025":False,
    },
    {
        "item":"ensemble weights",
        "value":"OOF SSE minimization 2021-2023",
        "uses_2025":False,
    },
    {
        "item":"DOC weights",
        "value":str(DOC_WEIGHTS),
        "uses_2025":False,
    },
    {
        "item":"budget",
        "value":BUDGET_MAX,
        "uses_2025":False,
    },
    {
        "item":"formations",
        "value":";".join(FORMACOES.keys()),
        "uses_2025":False,
    },
    {
        "item":"primary contrast",
        "value":"FAME-DOC vs Expected-only normalized representation",
        "uses_2025":False,
    },
    {
        "item":"2025 role",
        "value":"sequential out-of-time evaluation; DOC not recalibrated",
        "uses_2025":False,
    },
])

protocol.to_csv(
    AUDIT_DIR / "experimental_protocol_freeze_audit.csv",
    index=False,
    encoding="utf-8-sig"
)

catalog_rows=[
    ("availability_tuning_summary.csv","Methodology","Availability model tuning"),
    ("frozen_predictive_hyperparameters.csv","Methodology","Temporal RF/XGB tuning"),
    ("frozen_ensemble_weights.csv","Methodology","OOF-derived ensemble weights"),
    ("predictive_metrics_separated_estimands.csv","Results","Predictive evaluation"),
    ("predictive_metrics_common_sample.csv","Results","Predictive common-sample comparison"),
    ("availability_probability_evaluation.csv","Results","Availability calibration"),
    ("doc_calibration_summary.csv","Results","2024 DOC calibration"),
    ("frozen_doc_weights.csv","Methodology/Results","Weights frozen before 2025"),
    ("doc_calibration_bootstrap_stability.csv","Results","Calibration stability"),
    ("operational_summary_2025.csv","Results","Out-of-time operational performance"),
    ("primary_doc_vs_expected_normalized_inference.csv","Results","Primary scientific contrast"),
    ("pairwise_wilcoxon_holm.csv","Results","Secondary multiplicity-controlled comparisons"),
    ("missing_next_market_sensitivity_summary.csv","Sensitivity","Target convention robustness"),
    ("decision_component_correlations.csv","Results","Component redundancy audit"),
    ("scout_consistency_audit_2021_2025.csv","Methodology","Scout harmonization audit"),
    ("ranking_metrics_summary.csv","Results","Ranking/representation quality"),
    ("posthoc_2025_weight_landscape.csv","Results","Temporal representation shift"),
    ("lineup_similarity_doc_comparators.csv","Results","Discrete decision consequences"),
    ("ablation_summary_2025.csv","Results","Ablation"),
    ("captain_summary.csv","Results","Captain analysis"),
    ("experimental_protocol_freeze_audit.csv","Reproducibility","Frozen experimental protocol"),
]

evidence_catalog=pd.DataFrame(
    catalog_rows,
    columns=["file","section","supports"]
)

evidence_catalog.to_csv(
    ROOT / "evidence_catalog.csv",
    index=False,
    encoding="utf-8-sig"
)

manifest=[]

for p in sorted(ROOT.rglob("*.csv")):

    if p.name=="manifest_sha256.csv":
        continue

    try:
        df=pd.read_csv(p)
        n_rows,n_cols=df.shape
    except Exception:
        n_rows,n_cols=np.nan,np.nan

    manifest.append({
        "relative_path":str(p.relative_to(ROOT)),
        "n_rows":n_rows,
        "n_columns":n_cols,
        "size_bytes":p.stat().st_size,
        "sha256":hashlib.sha256(p.read_bytes()).hexdigest(),
    })

manifest=pd.DataFrame(manifest)

manifest.to_csv(
    ROOT / "manifest_sha256.csv",
    index=False,
    encoding="utf-8-sig"
)

display(protocol)

print(
    "Pipeline v11.1 concluído. Outputs em:",
    ROOT.resolve()
)
